# 🎯 [K14_M06] 중대별 비교하기
### 실습 솔루션 & 정답 해설 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/k14_m06_solution.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 병영생활·부대관리 (Level 1) / K14 군 생활 데이터 분석 |
| **미션** | 미션 06. 중대별 비교하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 군 생활 데이터 분석, HITL 수기 검산, 시각화 왜곡 방지 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



---

## 🎯 미션 06. 중대별 비교하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **중대별 차이를 공정하게 비교합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **인원이 다른 3개 중대를 총합이 아니라 1인당 값으로 비교하게 하고, 총합과 1인당 값의 순위가 달라지는 항목이 있는지 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `중대별 1인당 비교표, 주목할 항목 목록` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 6 1인당 비교표

| **항목** | **1중대(1인당)** | **2중대(1인당)** | **3중대(1인당)** | **총합 순위와 달라졌는가** |
| --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 비교 분석 — 인원 차이를 고려한 기준으로 중대별 차이를 비교하고 주목 항목을 골랐는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(중대별 1인당 비교표, 주목할 항목 목록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 06.] 정답 해설 - 모범 프롬프트 및 완성 산출물 리포트
mission_06_prompt = """[인원 차이 보정 1인당 정규화 비교 및 순위 역전 분석 질의]
당신은 부대 통계 정규화 분석 전문가입니다. 3개 중대의 인원수(1중대 48명, 2중대 52명, 3중대 45명)가 서로 다르므로 단순 총합 비교 시 발생하는 통계적 왜곡을 방지하기 위해, 4주 총 이용량을 중대 인원수로 나눈 '1인당 4주 총 이용량 비교표'를 산출해 주십시오.
총합 기준 순위와 1인당 정규화 순위 간에 발생한 순위 역전 현상(예: 체력단련실 2중대 vs 3중대)을 분석하고, 부대 관찰이 시급한 중점 관리 대상 지표 2개를 선정해 주십시오."""

mission_06_result = """## 1. 1인당 4주 총 이용량 비교표
| 항목 | 1중대 (48명) | 2중대 (52명) | 3중대 (45명) | 최고 중대 (1인당) |
| :--- | :--- | :--- | :--- | :--- |
| 체력단련실 1인당 | 9.92회 (476/48) | 6.83회 (355/52) | 9.36회 (421/45) | 1중대 |
| 싸지방 1인당 | 4.98회 (239/48) | 2.12회 (110/52) | 4.40회 (198/45) | 1중대 |
| PX 1인당 | 11.56회 (555/48) | 12.98회 (675/52) | 10.58회 (476/45) | 2중대 |

## 2. 총합 순위 vs 1인당 순위 역전 분석표
| 항목 | 단순 총합 순위 | 1인당 평균 순위 | 역전 여부 및 원인 |
| :--- | :--- | :--- | :--- |
| 체력단련실 | 1위: 1중대(476), 2위: 3중대(421), 3위: 2중대(355) | 1위: 1중대(9.92), 2위: 3중대(9.36), 3위: 2중대(6.83) | 순위 유지 |
| 싸지방 | 1위: 1중대(239), 2위: 3중대(198), 3위: 2중대(110) | 1위: 1중대(4.98), 2위: 3중대(4.40), 3위: 2중대(2.12) | 순위 유지 (2중대 격차 극대화 확인) |
| PX 이용 | 1위: 2중대(675), 2위: 1중대(555), 3위: 3중대(476) | 1위: 2중대(12.98), 2위: 1중대(11.56), 3위: 3중대(10.58) | 순위 유지 (2중대의 압도적 PX 의존도 확인) |

## 3. 주목할 핵심 항목 2~3개 및 선정 사유
1. **2중대의 저조한 시설 이용률**: 체력단련실(6.8회), 싸지방(2.1회) 모두 타 중대 대비 1인당 이용량이 현저히 낮음.
2. **3중대의 건의 집중 현상**: 인원수는 가장 적으나(45명) 건의 건수는 33건으로 대대 전체의 47.8%를 차지함.

## 4. 도전 과제: 100명당 건의 발생 비율
- **1중대**: 25.0건 / 100명
- **2중대**: 46.2건 / 100명
- **3중대**: 73.3건 / 100명 (타 중대 대비 3배 가까운 체감 건의 강도)"""

# 렌더러 실행 (모범 해설 결과를 시각적으로 렌더링합니다)
show_result("미션 06. 중대별 비교하기", mission_06_prompt, mission_06_result)



In [ ]:
# [미션 06 실습 보조] 중대별 1인당 4주 총 이용 횟수 계산 (정규화)
personnel = df.groupby('중대')['인원(명)'].first()
totals = df.groupby('중대')[['체력단련실', '싸지방', 'PX', '건의']].sum()
per_capita = totals.div(personnel, axis=0).round(2)

print('=== 1인당 4주 총 이용 횟수 (총합 ÷ 인원수) ===')
display(per_capita[['체력단련실', '싸지방', 'PX']])

print('\n=== 장병 100명당 건의 발생률 ===')
suggestions_rate = (totals['건의'].div(personnel) * 100).round(1).to_frame(name='100명당 건의 건수')
display(suggestions_rate)